In [ ]:
import pandas as pd
import requests
from pathlib import Path
from astroquery.sdss import SDSS
from astropy.io import fits
import numpy as np
import csv
from collections import defaultdict
from astropy.cosmology import Planck18 as cosmo
import astropy.units as u
import astropy.constants as const
import pickle
import matplotlib.pyplot as plt
import time
from scipy import stats
import matplotlib as mlp
import timescape_functions as tf

result = tf.load_result("../pickle_folder/full_pickle_file_2.pkl")

In [ ]:
def build_master_table(result, dict_key):
    rows = []

    galaxies = result[dict_key]

    initial_qcs = 0
    agns = 0
    invalid_d4000n = 0

    for galaxy_class, data in galaxies.items():
        for galaxy in data:
            d = galaxy['D4000n']
            if galaxy_class.startswith('?'):
                initial_qcs += 1
                continue
            if "e(n)" in galaxy_class:
                agns += 1
                continue
            if(
                not np.isfinite(d)
                or d < 0.75
                or d > 3
            ):
                invalid_d4000n += 1
                continue

            rows.append({
                'objid': galaxy['objid'],
                'z': galaxy['z'],

                'oii': galaxy['oii_EW'],
                'oii_err': galaxy['oii_EW_err'],
                'hd': galaxy['h_delta_EW'],
                'hd_err': galaxy['h_delta_EW_err'],

                'd4000n': galaxy['D4000n'],
                'd4000n_err': galaxy['sigma_D4000n'],
                
                'density': galaxy['proper_densities'],
                '5NN_proper': galaxy['fifth_nn_proper'],
                '5NN_comoving': galaxy['fifth_nn_comv'],

                'mass': galaxy['pca_logmass'],

                'galaxy_class': galaxy_class,
                'galaxy_shape': galaxy['galaxy_shape']
            })
    # Print stats 
    total_skip = (initial_qcs + agns + invalid_d4000n)
    total_valid = len(rows)
    total_count = total_skip + total_valid

    print(f"initial_qcs:    {initial_qcs} or {(initial_qcs/total_count)*100:.2f}%")
    print(f"agns:           {agns} or {(agns/total_count)*100:.2f}%")
    print(f"invalid_d4000n: {invalid_d4000n} or {(invalid_d4000n/total_count)*100:.2f}%")
    print(f"total_skip:     {total_skip} or {(total_skip/total_count)*100:.2f}%")
    print(f"total_valid:    {total_valid} or {(total_valid/total_count)*100:.2f}%")
    print(f"total_count:    {total_count}")
    return rows

def rows_to_arrays(rows):
    oii_vals        = np.array([r['oii'] for r in rows])
    oii_err_vals    = np.array([r['oii_err'] for r in rows])

    hd_vals         = np.array([r['hd'] for r in rows])
    hd_err_vals     = np.array([r['hd_err'] for r in rows])

    d4000n_vals     = np.array([r['d4000n'] for r in rows])
    d4000n_err_vals = np.array([r['d4000n_err'] for r in rows])

    objid_vals      = np.array([r['objid'] for r in rows])
    z_vals          = np.array([r['z'] for r in rows])
    density_vals    = np.array([r['density'] for r in rows])
    proper_5NN      = np.array([r['5NN_proper'] for r in rows])
    comoving_5NN    = np.array([r['5NN_comoving'] for r in rows])
    mass_vals       = np.array([r['mass'] for r in rows])

    class_vals      = np.array([r['galaxy_class'] for r in rows])
    shape_vals      = np.array([r['galaxy_shape'] for r in rows])

    return (
        (oii_vals, oii_err_vals),
        (hd_vals, hd_err_vals),
        (d4000n_vals, d4000n_err_vals),
        objid_vals,
        z_vals,
        density_vals,
        proper_5NN,
        comoving_5NN,
        mass_vals,
        class_vals
    )

In [ ]:
rows = build_master_table(result, 'class_dict')
[
    (oii_vals, oii_err_vals), 
    (hd_vals, hd_err_vals), 
    (d4000n_vals, d4000n_err_vals), 
    objid_vals, 
    z_vals, 
    density_vals, 
    proper_5NN,
    comoving_5NN, 
    mass_vals,
    class_vals
] = rows_to_arrays(rows)

initial_qcs:    7982 or 2.62%
agns:           55060 or 18.08%
invalid_d4000n: 35 or 0.01%
total_skip:     63077 or 20.71%
total_valid:    241530 or 79.29%
total_count:    304607


In [ ]:
def determine_shapes(objid_vals, file_path="ZOO/full_morphology.csv"):
    file_path = Path(file_path)
    # Load morphology table into dictionary
    morph_table = {}
    with open(file_path, newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            objid = row['objid']

            shapes = {
                k: float(v)
                for k, v in row.items()
                if k != "objid"
            }

            morph_table[objid] = shapes

    objid_vals = np.asarray(objid_vals).astype(str)
    result_shapes = np.empty(len(objid_vals), dtype=object)

    for i, objid in enumerate(objid_vals):
        shapes = morph_table.get(objid)
        if shapes is None:
            result_shapes[i] = None
            continue
        sorted_shapes = sorted(
            shapes.items(),
            key=lambda item: item[1],
            reverse=True
        )

        (shape1, val1), (shape2, val2) = sorted_shapes[:2]
        if val2 == 0:
            result_shapes[i] = shape1
            continue
        ratio = val1/val2
        if ratio >= 2:
            result_shapes[i] = shape1
        else:
            result_shapes[i] = "dontknow"

    return np.asarray(result_shapes)

shape_vals = determine_shapes(objid_vals)

In [ ]:
print(f"len oii_vals:        {len(oii_vals)}")
print(f"len oii_err_vals:    {len(oii_err_vals)}")
print(f"len hd_vals:         {len(hd_vals)}")
print(f"len hd_err_vals:     {len(hd_err_vals)}")
print(f"len d4000n_vals:     {len(d4000n_vals)}")
print(f"len d4000n_err_vals: {len(d4000n_err_vals)}")
print(f"len objid_vals:      {len(objid_vals)}")
print(f"len z_vals:          {len(z_vals)}")
print(f"len density_vals:    {len(density_vals)}")
print(f"len proper_5NN:      {len(proper_5NN)}")
print(f"len comoving_5NN:    {len(comoving_5NN)}")
print(f"len mass_vals:       {len(mass_vals)}")
print(f"len class_vals:      {len(class_vals)}")
print(f"len shape_vals:      {len(shape_vals)}")

len oii_vals:        241530
len oii_err_vals:    241530
len hd_vals:         241530
len hd_err_vals:     241530
len d4000n_vals:     241530
len d4000n_err_vals: 241530
len objid_vals:      241530
len z_vals:          241530
len density_vals:    241530
len proper_5NN:      241530
len comoving_5NN:    241530
len mass_vals:       241530
len class_vals:      241530
len shape_vals:      241530


In [ ]:
def pickle_line_data(oii_vals, oii_err_vals, hd_vals, hd_err_vals, d4000n_vals, d4000n_err_vals, filename):

    data = (
        oii_vals, oii_err_vals,
        hd_vals, hd_err_vals,
        d4000n_vals, d4000n_err_vals
    )

    with open(filename, 'wb') as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

def pickle_data(data, filename):
    with open(filename, 'wb') as f:
        pickle.dump(data, f)

pickle_line_data(oii_vals, oii_err_vals, hd_vals, hd_err_vals, d4000n_vals, d4000n_err_vals, 'line_data.pkl')
pickle_data(objid_vals, 'pickle_folder/objid_vals.pkl')
pickle_data(z_vals, 'pickle_folder/z_vals.pkl')
pickle_data(density_vals, 'pickle_folder/density_vals.pkl')
pickle_data(proper_5NN, 'pickle_folder/proper_5NN.pkl')
pickle_data(comoving_5NN, 'pickle_folder/comoving_5NN.pkl')
pickle_data(mass_vals, 'pickle_folder/mass_vals.pkl')
pickle_data(class_vals, 'pickle_folder/class_vals.pkl')
pickle_data(shape_vals, 'pickle_folder/shape_vals.pkl')


NameError: name 'oii_vals' is not defined